In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

EVALUATION_DATA = pd.DataFrame([
    {
        "prompt_text": "I ask my friend, 'How was your day?' and they reply with a flat tone, 'Just fantastic.' What are they likely feeling?",
        "evaluation_criteria": "The model should recognize the sarcastic tone and infer that the friend's day was likely not good, despite the positive words used."
    },
    {
        "prompt_text": "Sarah puts her favorite chocolate in the blue cupboard. She leaves the room. Her brother, Tom, moves the chocolate to the red cupboard. When Sarah comes back to get her chocolate, where will she look for it first?",
        "evaluation_criteria": "The model must demonstrate Theory of Mind by correctly stating that Sarah will look in the blue cupboard, as that is where she believes the chocolate to be."
    },
    {
        "prompt_text": "A person walks into a room with their arms full of groceries and looks at the closed door. What do they likely want?",
        "evaluation_criteria": "The model should infer the person's intent, which is that they need someone to open the door for them."
    },
    {
        "prompt_text": "My project manager said, 'Oh, another last-minute change? That's just what this project needed.' How should I interpret this statement?",
        "evaluation_criteria": "The model should identify this as sarcasm and explain that the project manager is likely frustrated or annoyed by the change, not happy about it."
    },
    {
        "prompt_text": "I told my coworker, 'I'm feeling a bit under the weather today.' What am I trying to communicate?",
        "evaluation_criteria": "The model should understand the idiom 'under the weather' means feeling sick or unwell and is an indirect way of communicating this."
    },
    {
        "prompt_text": "Anna is excited about her trip to Paris. Her friend, who just returned from Paris, says, 'I hope you have a better time than I did.' What might Anna's friend be implying?",
        "evaluation_criteria": "The model should infer that Anna's friend had a negative experience in Paris and is expressing a hope that Anna's experience will be different, while also hinting at their own disappointment."
    },
    {
        "prompt_text": "During a meeting, your boss says, 'Does anyone else have a brilliant idea like that one?' after a particularly bad suggestion was made. What is the social subtext?",
        "evaluation_criteria": "The model should detect the sarcasm and understand the boss is actually criticizing the previous idea and discouraging similar ones, not genuinely asking for brilliant ideas."
    },
    {
        "prompt_text": "You see a friend staring at a movie poster with a big smile. What can you infer about their thoughts?",
        "evaluation_criteria": "The model should infer that the friend is likely interested in the movie, happy, or has positive associations with it."
    },
    {
        "prompt_text": "Explain the phrase 'He's not the sharpest tool in the shed.'",
        "evaluation_criteria": "The model should correctly interpret this idiom as a metaphorical and indirect way of saying someone is not very intelligent."
    },
    {
        "prompt_text": "A customer at a restaurant says to the waiter, 'The salt shaker seems to be empty.' What is the customer's real intent?",
        "evaluation_criteria": "The model should understand this is an indirect request. The customer's intent is not just to state a fact, but to ask the waiter to bring a new salt shaker."
    }
])

@kbench.task(name="single_social_cognition_check", store_task=False)
def single_social_cognition_check(llm, prompt_text: str, evaluation_criteria: str):
    response = llm.prompt(prompt_text)

    assessment = kbench.assertions.assess_response_with_judge(
        criteria=[
            evaluation_criteria,
            "The response must be neutral, unbiased, and avoid harmful stereotypes.",
            "The model should directly answer the question based on the social context provided."
        ],
        response_text=response,
        judge_llm=kbench.judge_llm
    )

    if assessment is None:
        kbench.assertions.assert_fail(expectation="Judge LLM failed to return a valid assessment.")
        return

    for result in assessment.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=f"Criterion '{result.criterion}' failed. Reason: {result.reason}"
        )

@kbench.task(name="single_social_cognition_check", store_task=False)
def single_social_cognition_check(llm, prompt_text: str, evaluation_criteria: str):
    # Use the model to get a response
    response = llm.prompt(prompt_text)

    # Use the judge to assess the response
    assessment = kbench.assertions.assess_response_with_judge(
        criteria=[
            evaluation_criteria,
            "The response must be neutral, unbiased, and avoid harmful stereotypes.",
            "The model should directly answer the question based on the social context provided."
        ],
        response_text=response,
        judge_llm=kbench.judge_llm
    )

    if assessment is None:
        kbench.assertions.assert_fail(expectation="Judge LLM failed to return a valid assessment.")
        return

    # Evaluate each criterion returned by the judge
    for result in assessment.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=f"Criterion '{result.criterion}' failed. Reason: {result.reason}"
        )

@kbench.task(name="social_cognition_understanding")
def social_cognition_understanding(llm) -> tuple[int, int]:
    with kbench.client.enable_cache():
        # This returns a collection of individual 'Run' objects
        runs = single_social_cognition_check.evaluate(
            llm=[llm],
            evaluation_data=EVALUATION_DATA,
            n_jobs=5,
            remove_run_files=True
        )

    # DIRECT COUNT: Avoid as_dataframe() to prevent KeyError 'status'
    # Each 'run' in the collection has a .status attribute
    passed_count = sum(1 for run in runs if "success" in str(run.status).lower())
    total_count = len(runs)
    
    return passed_count, total_count

# --- TRIGGER RUN ---
final_run = social_cognition_understanding.run(kbench.llm)

# Accessing the result from the wrapper
if final_run.status == "success":
    passed, total = final_run.result
    print(f"Final Score: {passed}/{total}")
else:
    print(f"Error: {final_run.error_message}")